# 📊 KN — Análisis y predicción de incidentes de mantención

Notebook para analizar el export de tickets de **Fractal** (`EXPORTFRACTAL*.csv` / `.xlsx`) y anticipar fallas para mejorar el **SLA** de Kuehne + Nagel.

**Cómo usarlo:** `Entorno de ejecución → Ejecutar todo` y sube el archivo cuando lo pida la celda 1.

| # | Pregunta | Sección |
|---|----------|---------|
| 1 | ¿Dónde están los mayores incidentes? | 3 |
| 2 | ¿Cuáles son los activos más recurrentes? | 4 |
| 3 | ¿Cuáles son los problemas más recurrentes? | 5 |
| 4 | ¿Cuánto nos está costando? | 6 |
| 5 | ¿Cuánto nos demoramos en arreglar? (SLA) | 7 |
| 6 | ¿Qué usuario levanta más tickets? | 8 |
| 🔮 | **Predictivo:** volumen futuro, activos que van a fallar, riesgo de incumplir SLA, problemas en alza | 9 |
| 📋 | Resumen ejecutivo + descarga en Excel | 10 |

## ⚙️ 0. Configuración

In [ ]:
# ---- Parámetros editables ----
SLA_HORAS = 216            # umbral SLA observado en el export (Ontime ≤ ~9 días corridos)
TOP_N = 15                 # cuántos elementos mostrar en los rankings
SEMANAS_PRONOSTICO = 8     # horizonte del pronóstico de volumen
DIAS_ALERTA_ACTIVOS = 30   # ventana para "activos que probablemente fallen pronto"
MIN_INCIDENTES_ACTIVO = 3  # mínimo de incidentes para calcular MTBF de un activo
EXCLUIR_CANCELADOS = True  # excluir tickets cancelados / duplicados / de prueba de los análisis
RUTA_ARCHIVO = None        # opcional: ruta fija (ej. '/content/drive/MyDrive/EXPORTFRACTAL1.csv')

import os, re, io, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams.update({'figure.figsize': (11, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3})
COLOR = '#1f4e79'; COLOR_ALERTA = '#c0392b'

try:
    import google.colab  # noqa
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
RUTA_ARCHIVO = RUTA_ARCHIVO or os.environ.get('KN_ARCHIVO')

## 📥 1. Cargar el archivo

In [ ]:
def leer_archivo(nombre, contenido=None):
    fuente = io.BytesIO(contenido) if contenido is not None else nombre
    if nombre.lower().endswith(('.xlsx', '.xls')):
        return pd.read_excel(fuente)
    for enc in ('utf-8-sig', 'latin-1'):
        for sep in (',', ';'):
            try:
                if hasattr(fuente, 'seek'): fuente.seek(0)
                d = pd.read_csv(fuente, encoding=enc, sep=sep)
                if d.shape[1] > 5:
                    return d
            except UnicodeDecodeError:
                break
            except Exception:
                continue
    raise ValueError('No se pudo leer el archivo')

if RUTA_ARCHIVO:
    raw = leer_archivo(RUTA_ARCHIVO)
    print('Archivo:', RUTA_ARCHIVO)
elif EN_COLAB:
    from google.colab import files
    subido = files.upload()
    nombre = list(subido)[0]
    raw = leer_archivo(nombre, subido[nombre])
    print('Archivo:', nombre)
else:
    raise SystemExit('Define RUTA_ARCHIVO o ejecuta en Colab')
print(f'{raw.shape[0]:,} filas x {raw.shape[1]} columnas')

## 🧹 2. Limpieza y preparación

* Normaliza nombres de columnas (tildes / encoding).
* **Corrige el desfase de encabezados** del export de Fractal: desde *Clasificación 2* los datos están corridos una columna (ej. *Clasificación 2* trae la prioridad y *Email del solicitante* trae el técnico).
* Clasifica cada ticket en una **categoría de problema** usando palabras clave de la descripción.

In [ ]:
def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s).strip().lower()

df = raw.copy()
df.columns = [norm(c) for c in df.columns]
df = df.loc[:, ~df.columns.str.startswith('unnamed')]

# --- Corrección de desfase de encabezados (Clasificación 2 -> ID del Estado) ---
prio_vals = set(df.get('clasificacion 2', pd.Series(dtype=str)).dropna().astype(str).str.strip().str.lower())
if 'prioridad' in df and df['prioridad'].isna().all() and prio_vals & {'alta', 'media', 'baja'}:
    df = df.rename(columns={
        'clasificacion 2': 'prioridad_real',
        'calificacion -> notas': 'creada_por_real',
        'creada por': 'solicitante_real',
        'solicitado por': 'email_solicitante_real',
        'email del solicitante': 'tecnico_real',
        'palabras clave': 'id_estado_real',
        'id del estado': 'palabras_clave_real',
    })
    df = df.dropna(axis=1, how='all')  # quita las columnas vacías que quedaron con el nombre original
    print('✔ Desfase de encabezados detectado y corregido')
    MAP = {'prioridad_real': 'prioridad', 'solicitante_real': 'solicitante',
           'email_solicitante_real': 'email_solicitante', 'tecnico_real': 'tecnico',
           'palabras_clave_real': 'palabras_clave'}
else:
    MAP = {'prioridad': 'prioridad', 'solicitado por': 'solicitante',
           'email del solicitante': 'email_solicitante', 'actualizado por': 'tecnico',
           'palabras clave': 'palabras_clave'}
MAP.update({'nro solicitud': 'id', 'orden de trabajo': 'ot', 'estado': 'estado',
            'descripcion': 'descripcion', 'observaciones': 'observaciones',
            'nota cambio de estado': 'nota_estado', 'es urgente?': 'urgente',
            'activo': 'activo', 'codigo del activo': 'codigo_activo',
            'ubicado en o es parte de': 'ubicacion', 'fecha de creacion': 'fecha_creacion',
            'fecha cierre solicitud': 'fecha_cierre', 'tiempo transcurrido': 'horas_resolucion',
            'cumplimiento': 'cumplimiento', 'column40': 'sede', 'cost avoidance': 'costo',
            'dia de semana': 'dia_semana'})
df = df.rename(columns={k: v for k, v in MAP.items() if k in df})
df['urgente'] = df.get('urgente', pd.Series(index=df.index, dtype=str)).astype(str).str.strip()

# --- Tipos ---
df['fecha_creacion'] = pd.to_datetime(df['fecha_creacion'], errors='coerce')
df['fecha_cierre'] = pd.to_datetime(df['fecha_cierre'].replace('--', np.nan), errors='coerce')
df['horas_resolucion'] = pd.to_numeric(df['horas_resolucion'], errors='coerce')
df['dias_resolucion'] = df['horas_resolucion'] / 24
df['costo'] = pd.to_numeric(df['costo'].astype(str).str.replace(r'[^\d]', '', regex=True), errors='coerce')
df['cumplimiento'] = df['cumplimiento'].astype(str).str.strip()
df['sla_incumplido'] = (df['cumplimiento'].str.lower() == 'offtime').astype(int)

# --- Ubicación: sede / área ---
partes = df['ubicacion'].fillna('').str.split('/').apply(lambda p: [x.strip() for x in p if x.strip()])
sede_ruta = partes.apply(lambda p: p[1] if len(p) > 1 else np.nan)
if 'sede' in df:
    df['sede'] = df['sede'].astype(str).str.replace('/', '').str.strip().replace({'nan': np.nan})
    df['sede'] = df['sede'].fillna(sede_ruta)
else:
    df['sede'] = sede_ruta
df['sede'] = df['sede'].fillna('SIN SEDE').str.upper().replace({'MIRAFLORES': 'MIRAFLOR', 'BODEGA 3M': '3M'})
df['area'] = partes.apply(lambda p: ' / '.join(p[1:3]) if len(p) > 2 else (p[1] + ' / (general)' if len(p) > 1 else 'SIN UBICACIÓN'))

# --- Activo / personas ---
df['activo'] = df['activo'].fillna('SIN ACTIVO').str.replace(r'\s+', ' ', regex=True).str.strip()
df['codigo_activo'] = df['codigo_activo'].fillna('SIN ACTIVO').str.strip()
limpiar_nombre = lambda s: s.astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title().replace({'Nan': np.nan})
df['solicitante'] = limpiar_nombre(df['solicitante']).fillna('Sin solicitante')
df['tecnico'] = limpiar_nombre(df['tecnico']).fillna('Sin técnico')
df['email_solicitante'] = df['email_solicitante'].astype(str).str.strip().str.lower().replace({'nan': np.nan})
# unifica variantes del mismo usuario con el mismo email (ej. "Tiare" → "Tiare Vargas")
def completar_nombres(g):
    nombres = g['solicitante'].value_counts()
    return g['solicitante'].map(lambda n: next((m for m in nombres.index if m.startswith(n + ' ')), n))
con_email = df['email_solicitante'].notna()
df.loc[con_email, 'solicitante'] = df[con_email].groupby('email_solicitante', group_keys=False).apply(completar_nombres)

# --- Calendario ---
df['mes'] = df['fecha_creacion'].dt.to_period('M').astype(str)
df['semana'] = df['fecha_creacion'].dt.to_period('W-SUN').dt.start_time
df['hora'] = df['fecha_creacion'].dt.hour
DIAS = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']
df['dia_semana'] = df['fecha_creacion'].dt.dayofweek.map(dict(enumerate(DIAS)))

# --- Categoría del problema (palabras clave; gana la categoría con más coincidencias) ---
CATEGORIAS = {
    'Iluminación':                 r'\bluz\b|luces|lumin|foco|ilumin|\bled\b|ampolleta|tubo fluor|reflector',
    'Eléctrico':                   r'enchufe|electric|cable|tablero|energia|voltaje|automatico|interruptor|corte de luz|diferencial|transformador',
    'Climatización':               r'aire acond|climatiz|\bclima|split|calefac|ventilad|extractor|temperatura|condensa|\bfrio\b|calor',
    'Sanitario / Baños':           r'bano|\bwc\b|taza|cadena|estanque|urinario|lavamanos|confort|dispensador|tapad|obstru|destap|sifon|ducha|camarin',
    'Filtraciones / Agua':         r'fuga|filtra|gotera|llave de agua|grifo|red humeda|caneria|agua|lluvia|inund|humedad',
    'Puertas / Cerraduras / Portones': r'puerta|chapa|manilla|cerradura|porton|cortina metal|bisagra|pestillo|picaporte|brazo|barrera',
    'Andenes / Equipos logísticos': r'anden|rampa|plataforma|nivelador|cinta|transportadora|enzunchadora|jaula|rack|conveyor|traspaleta|transpaleta|balanza|bascula|sorter|carro|pantografo',
    'Maquinaria (grúas / baterías / termos)': r'grua|montacarg|horquilla|bateria|cargador|rueda|hidraulic|bocina|maquina|thermo|termo|motor',
    'Seguridad / CCTV / Incendio': r'cctv|camara|alarma|extintor|incendio|senaletica|emergencia|sensor|detector|insegur',
    'Infraestructura / Obra civil': r'techo|cielo|muro|pared|pintura|vidrio|ventana|baldosa|cubierta|reja|canaleta|pavimento|grieta|hoyo|estructura|oxido|piso danad|piso levant|piso quebr|ceramic|losa|escalera|soldadura',
    'Mobiliario / Instalaciones':  r'instal|soporte|\btv\b|televis|pizarra|mueble|silla|escritorio|colgar|fabric|casillero|mesa|estante|cortina|persiana',
    'Olores / Plagas / Aseo':      r'olor|plaga|raton|rata|paloma|insecto|aseo|limpieza|basura|desague|alcantarill',
}
texto = (df['descripcion'].fillna('') + ' ' + df['observaciones'].fillna('')).map(norm)
PATRONES = {cat: re.compile(norm(p)) for cat, p in CATEGORIAS.items()}
def categorizar(t):
    hits = {cat: len(p.findall(t)) for cat, p in PATRONES.items()}
    mejor = max(hits, key=hits.get)  # empate → la que aparece primero en CATEGORIAS
    return mejor if hits[mejor] else 'Otros'
df['categoria'] = texto.map(categorizar)
df['texto'] = texto

# --- Tickets válidos (sin cancelados / duplicados / pruebas) ---
es_prueba = df['descripcion'].fillna('').map(norm).str.fullmatch(r'(test|prueba)\W*')
es_dup = df['nota_estado'].fillna('').map(norm).str.contains('duplicad')
es_cancel = df['estado'].fillna('').map(norm).str.contains('cancel')
df['valido'] = ~(es_prueba | es_dup | (es_cancel if EXCLUIR_CANCELADOS else False))
d = df[df['valido']].copy()
cerrados = d[d['fecha_cierre'].notna() & d['horas_resolucion'].notna()]
FECHA_REF = df['fecha_creacion'].max().normalize()

display(Markdown(f'''
**Periodo:** {df.fecha_creacion.min():%d-%m-%Y} → {df.fecha_creacion.max():%d-%m-%Y}
**Tickets totales:** {len(df):,} · **válidos para análisis:** {len(d):,}
(excluidos: {int(es_prueba.sum())} pruebas, {int(es_dup.sum())} duplicados, {int(es_cancel.sum())} cancelados)
**Abiertos / en proceso:** {int(d.fecha_cierre.isna().sum())}
'''))
d[['id', 'estado', 'sede', 'area', 'activo', 'categoria', 'solicitante', 'tecnico', 'horas_resolucion', 'cumplimiento', 'costo']].head()

In [ ]:
# Helper de gráficos
def barras(serie, titulo, xlabel='', fmt='{:,.0f}', color=COLOR, n=TOP_N):
    s = serie.head(n)[::-1]
    ax = s.plot.barh(color=color, figsize=(11, max(3, .38 * len(s))))
    ax.set_title(titulo, fontsize=13, weight='bold'); ax.set_xlabel(xlabel); ax.set_ylabel('')
    for i, v in enumerate(s.values):
        ax.text(v, i, ' ' + fmt.format(v), va='center', fontsize=9)
    plt.tight_layout(); plt.show()

## 📍 3. ¿Dónde están los mayores incidentes?

In [ ]:
por_sede = d.groupby('sede').agg(tickets=('id', 'count'), urgentes=('urgente', lambda s: (s.str.lower() == 'si').sum()),
                                 pct_sla_incumplido=('sla_incumplido', 'mean'), costo_total=('costo', 'sum'),
                                 horas_mediana=('horas_resolucion', 'median')).sort_values('tickets', ascending=False)
por_sede['pct_del_total'] = por_sede['tickets'] / por_sede['tickets'].sum()
por_sede['pct_sla_incumplido'] *= 100; por_sede['pct_del_total'] *= 100
display(por_sede)
barras(por_sede['tickets'], 'Tickets por sede', 'tickets')

por_area = d.groupby('area').size().sort_values(ascending=False)
barras(por_area, f'Top {TOP_N} áreas / pisos con más incidentes', 'tickets')

In [ ]:
# Mapa de calor sede x categoría
piv = pd.crosstab(d['sede'], d['categoria'])
piv = piv.loc[piv.sum(1).sort_values(ascending=False).index, piv.sum().sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(14, .6 * len(piv) + 2))
im = ax.imshow(piv.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=40, ha='right')
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index); ax.grid(False)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        ax.text(j, i, v, ha='center', va='center', fontsize=9, color='white' if v > piv.values.max() * .5 else 'black')
ax.set_title('Incidentes por sede y tipo de problema', fontsize=13, weight='bold'); plt.colorbar(im); plt.tight_layout(); plt.show()

# Cuándo ocurren (día x hora)
dh = pd.crosstab(d['dia_semana'], d['hora']).reindex(DIAS).fillna(0)
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(dh.values, cmap='Oranges', aspect='auto'); ax.grid(False)
ax.set_yticks(range(7)); ax.set_yticklabels(DIAS); ax.set_xticks(range(dh.shape[1])); ax.set_xticklabels(dh.columns)
ax.set_xlabel('hora de creación'); ax.set_title('¿Cuándo se reportan los incidentes? (día x hora)', fontsize=13, weight='bold')
plt.colorbar(im); plt.tight_layout(); plt.show()

## 🏭 4. ¿Cuáles son los activos más recurrentes?

In [ ]:
act = d[d['codigo_activo'] != 'SIN ACTIVO']
top_activos = act.groupby(['activo', 'sede']).agg(
    tickets=('id', 'count'), primer=('fecha_creacion', 'min'), ultimo=('fecha_creacion', 'max'),
    categoria_principal=('categoria', lambda s: s.value_counts().index[0]),
    horas_mediana=('horas_resolucion', 'median'), costo_total=('costo', 'sum'),
    pct_sla_incumplido=('sla_incumplido', 'mean')).sort_values('tickets', ascending=False)
top_activos['pct_sla_incumplido'] *= 100
print(f'{d.codigo_activo.eq("SIN ACTIVO").sum()} tickets no tienen activo asignado')
display(top_activos.head(TOP_N))
barras(top_activos['tickets'].droplevel(1), f'Top {TOP_N} activos con más incidentes', 'tickets')

# Activo + tipo de problema que se repite (el mismo problema en el mismo lugar)
rep = act.groupby(['activo', 'categoria']).size().sort_values(ascending=False).rename('veces')
display(Markdown('**Mismo problema repetido en el mismo activo (candidatos a mantención preventiva):**'))
display(rep[rep >= 3].head(TOP_N).to_frame())

## 🔧 5. ¿Cuáles son los problemas más recurrentes?

In [ ]:
cat = d.groupby('categoria').agg(tickets=('id', 'count'), costo_total=('costo', 'sum'),
                                 horas_mediana=('horas_resolucion', 'median'),
                                 pct_sla_incumplido=('sla_incumplido', 'mean')).sort_values('tickets', ascending=False)
cat['pct_del_total'] = cat['tickets'] / cat['tickets'].sum() * 100
cat['pct_sla_incumplido'] *= 100
display(cat)
barras(cat['tickets'], 'Tickets por tipo de problema', 'tickets')

# Palabras / frases más frecuentes en las descripciones
from sklearn.feature_extraction.text import CountVectorizer
STOP = ('de la el en los las del se y a un una por con para que no al es lo su esta favor sector se encuentra '
        'observaciones solicita solicito realizar hay ya muy mal malo mala').split()
cv = CountVectorizer(ngram_range=(1, 2), stop_words=STOP, min_df=3, token_pattern=r'(?u)\b[a-z]{3,}\b')
X = cv.fit_transform(d['descripcion'].fillna('').map(norm))
frases = pd.Series(np.asarray(X.sum(0)).ravel(), index=cv.get_feature_names_out()).sort_values(ascending=False)
barras(frases, f'Top {TOP_N} palabras / frases en las descripciones', 'menciones', color='#2e86ab')

display(Markdown('**Revisa los "Otros" para ajustar el diccionario `CATEGORIAS` si hace falta:**'))
display(d.loc[d.categoria == 'Otros', 'descripcion'].head(15).to_frame())

## 💰 6. ¿Cuánto nos está costando?
Se usa la columna **Cost Avoidance** del export (monto asociado a cada OT, en CLP).

In [ ]:
clp = lambda x: f'${x:,.0f}'.replace(',', '.')
con_costo = d[d['costo'].notna()]
display(Markdown(f'''
* **Costo total:** {clp(d.costo.sum())} en {len(con_costo):,} tickets con monto ({len(con_costo)/len(d):.0%} de los válidos)
* **Costo promedio por ticket:** {clp(con_costo.costo.mean())} · mediana {clp(con_costo.costo.median())}
* **Costo mensual promedio:** {clp(d.groupby('mes').costo.sum().mean())}
'''))
barras(d.groupby('sede')['costo'].sum().sort_values(ascending=False), 'Costo por sede (CLP)', fmt='${:,.0f}')
barras(d.groupby('categoria')['costo'].sum().sort_values(ascending=False), 'Costo por tipo de problema (CLP)', fmt='${:,.0f}')
barras(top_activos['costo_total'].droplevel(1).sort_values(ascending=False), f'Top {TOP_N} activos más caros (CLP)', fmt='${:,.0f}', color=COLOR_ALERTA)

mensual = d.groupby('mes').agg(costo=('costo', 'sum'), tickets=('id', 'count'))
fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.bar(mensual.index, mensual['costo'], color=COLOR, alpha=.8, label='costo')
ax1.set_ylabel('costo (CLP)'); ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx(); ax2.plot(mensual.index, mensual['tickets'], color=COLOR_ALERTA, marker='o', label='tickets'); ax2.set_ylabel('tickets'); ax2.grid(False)
ax1.set_title('Costo y volumen de tickets por mes', fontsize=13, weight='bold'); fig.legend(loc='upper left'); plt.tight_layout(); plt.show()

# Pareto: ¿qué % de activos concentra el 80% del costo?
pc = act.groupby('activo')['costo'].sum().sort_values(ascending=False)
acum = pc.cumsum() / pc.sum()
n80 = int((acum <= .8).sum()) + 1
print(f'Pareto: {n80} de {len(pc)} activos ({n80/len(pc):.0%}) concentran el 80% del costo asignado a activos.')

## ⏱️ 7. ¿Cuánto nos demoramos en arreglar? (tiempos y SLA)

In [ ]:
def resumen_tiempos(g):
    return cerrados.groupby(g).agg(tickets=('id', 'count'), horas_mediana=('horas_resolucion', 'median'),
                                   horas_promedio=('horas_resolucion', 'mean'),
                                   horas_p90=('horas_resolucion', lambda s: s.quantile(.9)),
                                   pct_sla_cumplido=('sla_incumplido', lambda s: 100 * (1 - s.mean())))

display(Markdown(f'''
* **Tiempo mediano de resolución:** {cerrados.horas_resolucion.median():.1f} h ({cerrados.dias_resolucion.median():.1f} días)
* **Tiempo promedio:** {cerrados.horas_resolucion.mean():.1f} h ({cerrados.dias_resolucion.mean():.1f} días) — el promedio está inflado por casos largos
* **90% de los tickets se cierra en menos de:** {cerrados.horas_resolucion.quantile(.9):.0f} h ({cerrados.dias_resolucion.quantile(.9):.1f} días)
* **Cumplimiento SLA global:** {100*(1-cerrados.sla_incumplido.mean()):.1f}%
'''))
for g, t in [('sede', 'sede'), ('categoria', 'tipo de problema'), ('tecnico', 'técnico'), ('prioridad', 'prioridad'), ('urgente', '¿es urgente?')]:
    display(Markdown(f'**Tiempos por {t}**'))
    display(resumen_tiempos(g).sort_values('horas_mediana', ascending=False))

barras(resumen_tiempos('categoria')['horas_mediana'].sort_values(ascending=False), 'Horas medianas de resolución por tipo de problema', 'horas', fmt='{:,.0f} h')
ta = cerrados[cerrados.codigo_activo != 'SIN ACTIVO'].groupby('activo').filter(lambda x: len(x) >= 2).groupby('activo')['horas_resolucion'].median().sort_values(ascending=False)
barras(ta, f'Top {TOP_N} activos que más demoran en repararse (mediana, ≥2 tickets)', 'horas', fmt='{:,.0f} h', color=COLOR_ALERTA)

ev = cerrados.groupby('mes').agg(horas_mediana=('horas_resolucion', 'median'), pct_sla=('sla_incumplido', lambda s: 100 * (1 - s.mean())))
fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.plot(ev.index, ev['pct_sla'], marker='o', color=COLOR, label='% SLA cumplido'); ax1.set_ylabel('% SLA cumplido'); ax1.set_ylim(0, 105)
ax1.tick_params(axis='x', rotation=45)
ax2 = ax1.twinx(); ax2.bar(ev.index, ev['horas_mediana'], alpha=.25, color=COLOR_ALERTA, label='horas medianas'); ax2.set_ylabel('horas medianas'); ax2.grid(False)
ax1.set_title('Evolución mensual del SLA y tiempo de resolución', fontsize=13, weight='bold'); fig.legend(loc='lower left'); plt.tight_layout(); plt.show()

## 🙋 8. ¿Qué usuario levanta más tickets?

In [ ]:
usr = df.groupby('solicitante').agg(
    tickets=('id', 'count'), validos=('valido', 'sum'),
    sede_principal=('sede', lambda s: s.value_counts().index[0]),
    problema_principal=('categoria', lambda s: s.value_counts().index[0]),
    urgentes=('urgente', lambda s: (s.str.lower() == 'si').sum())).sort_values('tickets', ascending=False)
usr['pct_cancelados_dup'] = 100 * (1 - usr['validos'] / usr['tickets'])
display(usr.head(TOP_N))
barras(usr['tickets'], f'Top {TOP_N} usuarios que levantan más tickets', 'tickets')
display(Markdown('_Un % alto de cancelados/duplicados indica oportunidad de capacitar en el uso de la plataforma._'))

display(Markdown('**Carga por técnico que atiende**'))
display(d.groupby('tecnico').agg(tickets=('id', 'count'), horas_mediana=('horas_resolucion', 'median'),
                                 pct_sla_incumplido=('sla_incumplido', lambda s: 100 * s.mean())).sort_values('tickets', ascending=False))

## 🔮 9. Análisis predictivo

1. **Pronóstico de volumen** de tickets para las próximas semanas (total y por sede) → dimensionar dotación.
2. **Activos con falla probable** en los próximos días (MTBF = tiempo medio entre fallas) → mantención preventiva.
3. **Modelo de riesgo de incumplir SLA** por ticket → priorizar los tickets con más riesgo apenas se crean.
4. **Problemas en alza** (últimos 90 días vs 90 días anteriores) → atacar la causa antes de que crezca.

### 9.1 Pronóstico de volumen semanal

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def pronosticar(serie, h=SEMANAS_PRONOSTICO):
    serie = serie.asfreq('W-MON', fill_value=0) if serie.index.freq is None else serie
    try:
        m = ExponentialSmoothing(serie, trend='add', damped_trend=True).fit()
        f = m.forecast(h).clip(lower=0)
        resid_sd = np.std(m.resid)
    except Exception:
        f = pd.Series([serie.tail(8).mean()] * h, index=pd.date_range(serie.index[-1] + pd.Timedelta(weeks=1), periods=h, freq='W-MON'))
        resid_sd = serie.tail(8).std()
    return f, resid_sd

sem = d.groupby('semana').size()
sem.index = pd.DatetimeIndex(sem.index)
sem = sem.asfreq('W-MON', fill_value=0).iloc[:-1]  # la última semana puede venir incompleta
f, sd = pronosticar(sem)
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(sem.index, sem.values, color=COLOR, label='real'); ax.plot(sem.index, sem.rolling(4).mean(), color='gray', ls='--', label='media móvil 4 sem')
ax.plot(f.index, f.values, color=COLOR_ALERTA, marker='o', label='pronóstico')
ax.fill_between(f.index, (f - 1.96 * sd).clip(lower=0), f + 1.96 * sd, color=COLOR_ALERTA, alpha=.15, label='intervalo 95%')
ax.set_title('Tickets por semana y pronóstico', fontsize=13, weight='bold'); ax.legend(); plt.tight_layout(); plt.show()

pron_sede = {}
for s_, g in d.groupby('sede'):
    ss = g.groupby('semana').size(); ss.index = pd.DatetimeIndex(ss.index)
    ss = ss.reindex(sem.index, fill_value=0)
    if ss.sum() >= 20:
        pron_sede[s_] = pronosticar(ss)[0].values
pron_sede = pd.DataFrame(pron_sede, index=f.index.date).T.round(1)
pron_sede['total_periodo'] = pron_sede.sum(1)
pron_sede['promedio_ult_8_sem'] = [d[(d.sede == s_) & (d.semana >= sem.index[-8])].shape[0] / 8 for s_ in pron_sede.index]
display(Markdown(f'**Tickets esperados por semana y sede (próximas {SEMANAS_PRONOSTICO} semanas):**'))
display(pron_sede.sort_values('total_periodo', ascending=False))

### 9.2 Activos con falla probable (mantención preventiva)
Para cada activo con al menos `MIN_INCIDENTES_ACTIVO` incidentes se calcula el **MTBF** (días promedio entre fallas).
**Próxima falla estimada = última falla + MTBF.** Si esa fecha ya pasó o cae dentro de `DIAS_ALERTA_ACTIVOS`, el activo entra en la lista de alerta.

In [ ]:
filas = []
for (a, s_), g in act.sort_values('fecha_creacion').groupby(['activo', 'sede']):
    if len(g) < MIN_INCIDENTES_ACTIVO:
        continue
    gaps = g['fecha_creacion'].diff().dt.total_seconds().dropna() / 86400
    gaps = gaps[gaps > 1]  # ignora tickets repetidos el mismo día
    if len(gaps) < 2:
        continue
    mtbf = gaps.median()
    ultima = g['fecha_creacion'].max()
    proxima = ultima + pd.Timedelta(days=mtbf)
    dias = (proxima - FECHA_REF).days
    cv = gaps.std() / gaps.mean() if gaps.mean() else np.nan  # regularidad (menor = más predecible)
    filas.append(dict(activo=a, sede=s_, incidentes=len(g), problema_frecuente=g['categoria'].mode()[0],
                      mtbf_dias=round(mtbf, 1), ultima_falla=ultima.date(), proxima_falla_estimada=proxima.date(),
                      dias_para_proxima=dias,
                      estado=('🔴 Atrasada (ya debió fallar)' if -mtbf <= dias < 0 else
                              f'🟠 Próximos {DIAS_ALERTA_ACTIVOS} días' if 0 <= dias <= DIAS_ALERTA_ACTIVOS else
                              '🟢 Más adelante' if dias > DIAS_ALERTA_ACTIVOS else '⚪ Sin fallas recientes'),
                      regularidad=round(1 / (1 + cv), 2) if pd.notna(cv) else np.nan,
                      costo_historico=g['costo'].sum()))
riesgo = pd.DataFrame(filas)
if len(riesgo):
    # Score 0-100: frecuencia, cercanía de la próxima falla, regularidad y costo
    r = lambda s: s.rank(pct=True)
    cercania = -riesgo.dias_para_proxima.abs()  # más cerca de la fecha estimada (antes o después) = más riesgo
    cercania[riesgo.estado.str.startswith('⚪')] = -9999
    riesgo['score_riesgo'] = (100 * (.35 * r(riesgo.incidentes) + .30 * r(cercania)
                                     + .15 * r(riesgo.regularidad.fillna(0)) + .20 * r(riesgo.costo_historico))).round(0)
    riesgo = riesgo.sort_values('score_riesgo', ascending=False)
    alerta = riesgo[riesgo.estado.str[0].isin(['🔴', '🟠'])]
    display(riesgo['estado'].value_counts().to_frame('activos'))
    display(Markdown(f'**{len(alerta)} activos en alerta (falla atrasada o esperada en ≤ {DIAS_ALERTA_ACTIVOS} días desde {FECHA_REF:%d-%m-%Y}) — programar preventiva:**'))
    display(alerta.head(25))
    barras(riesgo.set_index('activo')['score_riesgo'], f'Top {TOP_N} activos por score de riesgo (0-100)', 'score', color=COLOR_ALERTA)
else:
    print('No hay suficientes activos con historial para calcular MTBF')

### 9.3 Modelo de riesgo de incumplir SLA
Random Forest entrenado con la información disponible **al momento de crear el ticket** (sede, tipo de problema, activo, urgencia, prioridad, día/hora, historial del activo, técnico).
Se valida con un corte temporal (entrena con el pasado, evalúa con los tickets más recientes).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.inspection import permutation_importance

m = df[df['valido']].sort_values('fecha_creacion').copy()
# historial del activo hasta la fecha del ticket (sin mirar el futuro)
m['incidentes_previos_activo'] = m.groupby('codigo_activo').cumcount()
m.loc[m.codigo_activo == 'SIN ACTIVO', 'incidentes_previos_activo'] = 0
m['dias_desde_ultimo_activo'] = m.groupby('codigo_activo')['fecha_creacion'].diff().dt.days.fillna(365).clip(upper=365)
m['es_urgente'] = (m['urgente'].str.lower() == 'si').astype(int)
m['dow'] = m['fecha_creacion'].dt.dayofweek
m['mes_num'] = m['fecha_creacion'].dt.month
# carga de trabajo: tickets abiertos en el momento en que se crea cada ticket
m['backlog'] = [int(((m.fecha_creacion < t) & (m.fecha_cierre.isna() | (m.fecha_cierre > t))).sum()) for t in m.fecha_creacion]

CAT_COLS = ['sede', 'categoria', 'prioridad', 'tecnico']
NUM_COLS = ['es_urgente', 'hora', 'dow', 'mes_num', 'backlog', 'incidentes_previos_activo', 'dias_desde_ultimo_activo']

def features(x, columnas=None):
    X = pd.get_dummies(x[CAT_COLS].fillna('NA').astype(str), prefix=CAT_COLS).join(x[NUM_COLS].fillna(0))
    return X.reindex(columns=columnas, fill_value=0) if columnas is not None else X

entreno = m[m['fecha_cierre'].notna()]
corte = int(len(entreno) * .75)
tr, te = entreno.iloc[:corte], entreno.iloc[corte:]
Xtr = features(tr); Xte = features(te, Xtr.columns)
rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(Xtr, tr['sla_incumplido'])
p = rf.predict_proba(Xte)[:, 1]
try:
    auc = roc_auc_score(te['sla_incumplido'], p)
except ValueError:
    auc = np.nan
display(Markdown(f'**AUC en tickets recientes (validación temporal): {auc:.2f}** (0.5 = azar, 1 = perfecto)'))
if not auc >= .70:
    display(Markdown('> ⚠️ **Señal débil:** con los datos actuales, las características del ticket explican poco el incumplimiento del SLA; '
                     'el retraso depende más de la gestión (disponibilidad de técnicos, repuestos, proveedores). '
                     'Usa la probabilidad como apoyo y prioriza con la **tasa histórica sede × problema** (tabla de abajo). '
                     'El modelo mejora a medida que se acumulen más meses de datos y si se registran campos como *proveedor* o *requiere repuesto*.'))
print(classification_report(te['sla_incumplido'], (p >= .5).astype(int), target_names=['cumple', 'incumple'], zero_division=0))

pi = permutation_importance(rf, Xte, te['sla_incumplido'], n_repeats=10, random_state=42, scoring='roc_auc')
imp = pd.Series(pi.importances_mean, index=Xte.columns)
imp = imp.groupby(lambda c: next((k for k in CAT_COLS if c.startswith(k + '_')), c)).sum().sort_values(ascending=False)
barras(imp, '¿Qué variables explican el incumplimiento del SLA?', 'importancia (caída de AUC)', fmt='{:.3f}')

# Reentrena con todo el historial y puntúa los tickets abiertos
rf.fit(features(entreno), entreno['sla_incumplido'])
tasa_hist = entreno.groupby(['sede', 'categoria'])['sla_incumplido'].mean().rename('tasa_hist_incumple')
abiertos = m[m['fecha_cierre'].isna()].copy()
if len(abiertos):
    abiertos['prob_incumplir_sla'] = rf.predict_proba(features(abiertos, features(entreno).columns))[:, 1]
    abiertos = abiertos.join(tasa_hist, on=['sede', 'categoria'])
    abiertos['horas_abierto'] = (FECHA_REF + pd.Timedelta(days=1) - abiertos['fecha_creacion']).dt.total_seconds() / 3600
    display(Markdown('**Tickets abiertos ordenados por riesgo de incumplir SLA → atender primero:**'))
    display(abiertos.sort_values('prob_incumplir_sla', ascending=False)[
        ['id', 'fecha_creacion', 'sede', 'activo', 'categoria', 'tecnico', 'horas_abierto', 'prob_incumplir_sla', 'tasa_hist_incumple']])
else:
    print('No hay tickets abiertos para puntuar')

# Riesgo promedio por combinación sede x categoría (útil para reglas de priorización)
display(Markdown('**Combinaciones sede × problema con mayor tasa real de incumplimiento (≥5 tickets):**'))
comb = entreno.groupby(['sede', 'categoria']).agg(tickets=('id', 'count'), pct_incumple=('sla_incumplido', lambda s: 100 * s.mean()))
display(comb[comb.tickets >= 5].sort_values('pct_incumple', ascending=False).head(TOP_N))

### 9.4 Problemas en alza (últimos 90 días vs 90 días anteriores)

In [ ]:
ult = d[d.fecha_creacion > FECHA_REF - pd.Timedelta(days=90)]
ant = d[(d.fecha_creacion <= FECHA_REF - pd.Timedelta(days=90)) & (d.fecha_creacion > FECHA_REF - pd.Timedelta(days=180))]
def tendencia(col):
    t = pd.DataFrame({'ult_90d': ult.groupby(col).size(), 'prev_90d': ant.groupby(col).size()}).fillna(0).astype(int)
    t['variacion'] = t['ult_90d'] - t['prev_90d']
    t['variacion_pct'] = np.where(t['prev_90d'] > 0, 100 * t['variacion'] / t['prev_90d'].replace(0, np.nan), np.nan)
    return t.sort_values('variacion', ascending=False)
tend_cat = tendencia('categoria'); tend_sede = tendencia('sede'); tend_act = tendencia('activo')
display(Markdown('**Por tipo de problema**')); display(tend_cat)
display(Markdown('**Por sede**')); display(tend_sede)
display(Markdown('**Activos que más crecieron**')); display(tend_act[tend_act.variacion > 0].head(TOP_N))

tc = tend_cat['variacion']
ax = tc[::-1].plot.barh(color=[COLOR_ALERTA if v > 0 else COLOR for v in tc[::-1]], figsize=(11, 5))
ax.axvline(0, color='black', lw=.8); ax.set_title('Cambio en tickets por tipo de problema (últ. 90 días vs anteriores)', fontsize=13, weight='bold')
plt.tight_layout(); plt.show()

## 📋 10. Resumen ejecutivo y recomendaciones

In [ ]:
top_sede = por_sede.index[0]; top_cat = cat.index[0]
top_act = top_activos.index[0][0] if len(top_activos) else '-'
peor_sla = resumen_tiempos('categoria').query('tickets >= 10').sort_values('pct_sla_cumplido').index[0]
en_alza = tend_cat[tend_cat.variacion > 0].index.tolist()[:3]
n_alerta = len(alerta) if len(riesgo) else 0
display(Markdown(f'''
### Hallazgos
1. **Dónde:** {top_sede} concentra el {por_sede.loc[top_sede, "pct_del_total"]:.0f}% de los tickets; el área con más incidentes es **{por_area.index[0]}**.
2. **Activos:** el más recurrente es **{top_act}** ({top_activos.iloc[0]["tickets"]} tickets). {n80} activos concentran el 80% del costo.
3. **Problemas:** el principal es **{top_cat}** ({cat.loc[top_cat, "pct_del_total"]:.0f}% de los tickets).
4. **Costo:** {clp(d.costo.sum())} en el periodo (~{clp(d.groupby("mes").costo.sum().mean())} / mes).
5. **Tiempo:** mediana {cerrados.horas_resolucion.median():.0f} h; SLA cumplido {100*(1-cerrados.sla_incumplido.mean()):.0f}%. Peor SLA en **{peor_sla}**.
6. **Usuarios:** quien más tickets levanta es **{usr.index[0]}** ({usr.iloc[0]["tickets"]}).

### Acciones recomendadas para mejorar el SLA
* 🔧 **Plan preventivo** para los {n_alerta} activos en alerta (tabla 9.2) y para los pares *activo + problema* repetidos (sección 4).
* 🚨 **Priorizar al crear el ticket** los de combinaciones *sede × problema* con alta tasa histórica de incumplimiento y alta probabilidad del modelo (9.3).
* 📈 **Atacar la causa raíz** de los problemas en alza: {", ".join(en_alza) if en_alza else "sin alzas relevantes"}.
* 👷 **Dimensionar dotación** según el pronóstico semanal por sede (9.1).
* 🎓 **Capacitar** a los usuarios con alto % de tickets cancelados/duplicados y exigir activo asociado en cada ticket.
'''))

In [ ]:
# Exportar todos los resultados a Excel
salida = 'KN_resultados_incidentes.xlsx'
with pd.ExcelWriter(salida) as xw:
    por_sede.to_excel(xw, sheet_name='Sedes')
    por_area.rename('tickets').to_frame().to_excel(xw, sheet_name='Areas')
    top_activos.to_excel(xw, sheet_name='Activos')
    rep.to_frame().to_excel(xw, sheet_name='Activo_x_problema')
    cat.to_excel(xw, sheet_name='Problemas')
    resumen_tiempos('sede').to_excel(xw, sheet_name='Tiempos_sede')
    resumen_tiempos('categoria').to_excel(xw, sheet_name='Tiempos_problema')
    resumen_tiempos('tecnico').to_excel(xw, sheet_name='Tiempos_tecnico')
    usr.to_excel(xw, sheet_name='Usuarios')
    pron_sede.to_excel(xw, sheet_name='Pronostico_semanal')
    if len(riesgo): riesgo.to_excel(xw, sheet_name='Riesgo_activos', index=False)
    if len(abiertos): abiertos.sort_values('prob_incumplir_sla', ascending=False)[['id', 'fecha_creacion', 'sede', 'activo', 'categoria', 'tecnico', 'prob_incumplir_sla', 'tasa_hist_incumple']].to_excel(xw, sheet_name='Riesgo_abiertos', index=False)
    tend_cat.to_excel(xw, sheet_name='Tendencia_problemas')
    d.drop(columns=['texto']).to_excel(xw, sheet_name='Datos_limpios', index=False)
print('Excel generado:', salida)
if EN_COLAB:
    from google.colab import files
    files.download(salida)